# Exercise 4a: Rubric stress test (25 min)

1. **Write (8 min):** pick one failure mode from Exercise 3 and write its rubric entry from the template.
2. **Label (6 min):** two people label the same 12 traces **on their own**. No peeking, no discussing.
3. **Compare (8 min):** compute raw agreement and Cohen's kappa, then go through every disagreement and
   fix the rubric.
4. **Share (3 min):** one rubric change and what caused it.

In [ ]:
# Setup: run this cell first. It works in Google Colab and on your own laptop.
import os, sys
REPO_URL = "https://github.com/MarinaWyss/evaluating-ai-systems"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/EvalsWorkshop"):
        !git clone -q {REPO_URL} /content/EvalsWorkshop
        !pip install -q "litellm>=1.80.5" tenacity
    os.chdir("/content/EvalsWorkshop")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from beefcake import llm
llm.load_colab_secrets()
if llm.has_api_key():
    print("API key found. Bot model:", llm.get_model())
else:
    print("No API key found, so this notebook runs in offline mode with pre-generated data.")

## 1. Write your rubric

Open `rubric/rubric_template.md`, copy it, and fill it in. The worked example at the bottom shows a finished entry.

In [ ]:
print(open("rubric/rubric_template.md").read())

## 2. Label the 12 traces on your own

These are new traces you haven't seen. Label each one PASS or FAIL **for your failure mode only**.

In [ ]:
from beefcake.traces import load_traces, show_trace

traces = load_traces("exercise4a_traces.csv")
for i in range(len(traces)):
    show_trace(traces, i, show_history=False)

**Reviewer A** fills in this cell, and **Reviewer B** fills in the next one, each on their own laptop or without looking.

In [ ]:
reviewer_a = {
    "R01": "",
    "R02": "",
    "R03": "",
    "R04": "",
    "R05": "",
    "R06": "",
    "R07": "",
    "R08": "",
    "R09": "",
    "R10": "",
    "R11": "",
    "R12": "",
}

In [ ]:
reviewer_b = {
    "R01": "",
    "R02": "",
    "R03": "",
    "R04": "",
    "R05": "",
    "R06": "",
    "R07": "",
    "R08": "",
    "R09": "",
    "R10": "",
    "R11": "",
    "R12": "",
}

## 3. How much do you agree?

In [ ]:
from beefcake.evals import cohens_kappa, disagreements, raw_agreement

labels = traces[["Trace ID", "User Query", "AI Response"]].copy()
labels["A"] = labels["Trace ID"].map(reviewer_a).str.strip().str.upper()
labels["B"] = labels["Trace ID"].map(reviewer_b).str.strip().str.upper()
both = labels[labels["A"].isin(["PASS", "FAIL"]) & labels["B"].isin(["PASS", "FAIL"])]

if len(both) == 0:
    print("Fill in both reviewers' labels first, or run the demo at the bottom.")
else:
    print(f"Labeled by both: {len(both)} traces")
    print(f"Raw agreement:   {raw_agreement(both['A'], both['B']):.0%}")
    print(f"Cohen's kappa:   {cohens_kappa(both['A'], both['B']):.2f}")
    display(disagreements(both, "A", "B"))

**For every disagreement, ask:**

- Which part of the rubric caused it? A vague word in the definition? A missing edge case? No example like it?
- What rule or example would make the next decision obvious? Add it, bump the version, and write one line in the changelog.
- Still stuck? The owner of the rubric makes the call and writes down why.

Rough guide for kappa: above 0.6 is acceptable and above 0.8 is strong. These are conventions, not laws.

If you finish early, relabel the traces with your new version and see whether agreement goes up.

## Demo: what the numbers look like

No labels yet? Here are two example reviewers labeling the 25 traces from Module 2 for **Assumes device**.
Reviewer B passes the two borderline battery and heart-rate traces that Reviewer A fails.

In [ ]:
demo = load_traces("traces_v1_labeled.csv")[["Trace ID", "User Query", "AI Response", "Assumes device"]]
demo = demo.rename(columns={"Assumes device": "A"})
demo["B"] = demo["A"]
borderline = demo["User Query"].isin(["the battery dies really fast", "the heart rate numbers are way off"])
demo.loc[borderline, "B"] = demo.loc[borderline, "A"].map({"PASS": "FAIL", "FAIL": "PASS"})

print(f"Raw agreement: {raw_agreement(demo['A'], demo['B']):.0%}")
print(f"Cohen's kappa: {cohens_kappa(demo['A'], demo['B']):.2f}")
disagreements(demo, "A", "B")

Notice how 92% raw agreement becomes a much lower kappa. Most traces are an easy PASS, so agreeing on those is cheap. Kappa corrects for that.